### Setup

In [1]:
import json # !pip
import os
import textwrap
import pandas as pd
import importlib
import numpy as np
from pathlib import Path
import sys

In [2]:
ROOT_DIR = os.getcwd() + '/../'
sys.path.append(ROOT_DIR)

sys.path.append(ROOT_DIR+'src/')
print(ROOT_DIR)

/Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../


In [3]:
import src.annotate_scenario as annotate_scenario
import src.prompts as prompts
import src.translate_to_vis as translate_to_vis
import src.node as node
import src.get_emb_distances as get_emb_distances
import src.utils as utils
import src.moral_projection as moral_projection

import src.core_process as core_process
importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)
importlib.reload(prompts)
importlib.reload(node)
importlib.reload(utils)
importlib.reload(core_process)

<module 'src.core_process' from '/Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../src/core_process.py'>

In [4]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

### Overal Wrongness

In [5]:
# set main paths
SCENARIO_DIR = ROOT_DIR + "scenarios_inputs/" + "cheung_variants/"
# DATA_DIR_HUMAN = ROOT_DIR + "human_data/" 
OUTPUT_DIR = ROOT_DIR + "annotated_outputs/" + "cheung_variants/"

In [6]:
#set scenario file filename
SCENARIO_FILE = 'bird.json'


In [34]:
def get_score(scenario_file, scenario_id, act_id):
   
    #read in the scenario
    scenario_json = utils.open_scenario(SCENARIO_DIR, scenario_file, scenario_id, act_id)

    this_scenario = scenario_json['text']
    this_action = scenario_json['options'][act_id]



    scenario_deontology = scenario_json['deontology_level']
    scenario_utility = scenario_json['utility_level']

    this_text = this_scenario + ' I decide to ' + this_action + '.'

    # print(this_text)
    #call the process_values function to rate the moral goodness or wrongness of the action with no context

    resp = moral_projection.main([this_text])
    this_score = round(resp['projection'].iloc[0]*1000, 0)
    
    # print(this_score)
    output_dict = {'scenario_file': scenario_file, 'scenario_id': scenario_id, 'act_id': act_id,
                    'score': this_score, 'util': scenario_utility, 'deont': scenario_deontology}

    return output_dict


In [32]:
def collect_all_scores(scenario_file,n_scenarios):
    
    all_scores = []
    for scenario_id in range(1, n_scenarios+1):
        for act_id in ['1', '2']:
            output_dict = get_score(scenario_file, scenario_id, act_id)
            all_scores.append(output_dict)
    return pd.DataFrame(all_scores) 




In [35]:
all_scores = collect_all_scores(SCENARIO_FILE, 6)

When I was 9 or 10, I had a BB gun I would shoot in our backyard on weekends. One morning, I was
shooting at a target. When I was about to shoot, a bird started flying by. I didn’t notice the bird
and pulled the trigger as it entered my vision. What happened next seemed like slow motion. The bird
fell from flight and I started hearing it frantically wriggling in pain on the ground. I froze for a
minute as it slowly started to sink in that I had shot the bird. I frantically ran over. I felt
traumatized because I loved animals, especially birds, and I didn’t want to hurt them. I started to
cry because I didn’t know what to do next. I knew the bird was in pain and that it would not live. I
saw it dying a slow, painful death unless I acted. I had a shovel that I could use to hasten the
bird’s death. 


When I was 9 or 10, I had a BB gun I would shoot in our backyard on weekends. One morning, I was
shooting at a target. When I was about to shoot, a bird started flying by. I didn’t notice th

In [36]:
all_scores

,scenario_file,scenario_id,act_id,score,util,deont
0,bird.json,1,1,-28.0,3,1
1,bird.json,1,2,-25.0,3,1
2,bird.json,2,1,-35.0,3,2
3,bird.json,2,2,-24.0,3,2
4,bird.json,3,1,10.0,3,3
5,bird.json,3,2,-17.0,3,3
6,bird.json,4,1,-27.0,2,1
7,bird.json,4,2,-19.0,2,1
8,bird.json,5,1,-29.0,2,2
9,bird.json,5,2,-23.0,2,2


In [37]:
mean_by_util = all_scores.groupby('util', as_index=False)['score'].mean().rename(columns={'score': 'mean_score'})
mean_by_deont = all_scores.groupby('deont', as_index=False)['score'].mean().rename(columns={'score': 'mean_score'})
mean_by_util_deont = all_scores.groupby(['util', 'deont'], as_index=False)['score'].mean().rename(columns={'score': 'mean_score'})

print("Mean score by util level:")
display(mean_by_util)

print("Mean score by deont level:")
display(mean_by_deont)

print("Mean score by util + deont level:")
display(mean_by_util_deont)

Mean score by util level:


,util,mean_score
0,2,-17.166667
1,3,-19.833333


Mean score by deont level:


,deont,mean_score
0,1,-24.75
1,2,-27.75
2,3,-3.00


Mean score by util + deont level:


,util,deont,mean_score
0,2,1,-23.0
1,2,2,-26.0
2,2,3,-2.5
3,3,1,-26.5
4,3,2,-29.5
5,3,3,-3.5
